# Gradient Cobra

## Combine Classifier

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 455 samples, 30 features
Test set: 114 samples, 30 features


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from cobra.combine_classifier import CombineClassifier

model = CombineClassifier(
    estimators=[
        "logistic_regression",
        "random_forest",
        DecisionTreeClassifier(max_depth=5),
        GaussianNB()
    ],
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

/Users/ougi/Documents/Project/kfc-procedure/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [3]:
preds = model.predict(X_test)
accuracy = (preds == y_test).mean()
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.9561


In [4]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier


# =========================
# DATA
# =========================
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# ESTIMATOR POOLS
# =========================
pools = {
    "small": [
        LogisticRegression(max_iter=3000),
        RandomForestClassifier(n_estimators=200, random_state=42),
    ],
    "diverse": [
        RandomForestClassifier(n_estimators=200, random_state=42),
        SVC(probability=True),
        KNeighborsClassifier(n_neighbors=7),
    ],
    "strong": [
        RandomForestClassifier(n_estimators=300, random_state=42),
        SVC(probability=True),
        MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42),
    ]
}


# =========================
# CONFIG GRID
# =========================
configs = []

for pool_name, estimators in pools.items():
    for distance in ["hamming", "euclidean"]:
        for kernel in ["indicator", "rbf"]:
            for aggregator in ["majority_vote", "weighted_mean"]:
                configs.append({
                    "pool": pool_name,
                    "estimators": estimators,
                    "distance": distance,
                    "kernel": kernel,
                    "aggregator": aggregator
                })


# =========================
# RUN EXPERIMENTS
# =========================
results = []

for cfg in configs:
    model = CombineClassifier(
        estimators=cfg["estimators"],
        distance=cfg["distance"],
        kernel=cfg["kernel"],
        aggregator=cfg["aggregator"],
        random_state=42
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)

    results.append({
        "pool": cfg["pool"],
        "distance": cfg["distance"],
        "kernel": cfg["kernel"],
        "aggregator": cfg["aggregator"],
        "accuracy": acc
    })


# =========================
# SINGLE MODEL BASELINES
# =========================
baseline_models = {
    "LogReg": LogisticRegression(max_iter=3000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42)
}

for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        "pool": "single_model",
        "distance": "-",
        "kernel": "-",
        "aggregator": name,
        "accuracy": accuracy_score(y_test, pred)
    })


# =========================
# RESULTS TABLE
# =========================
df = pd.DataFrame(results)
df = df.sort_values(by="accuracy", ascending=False)

print(df.head(10))

ValueError: Classification metrics can't handle a mix of binary and continuous targets

# GradientCobra

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from cobra.gradientcobra import GradientCOBRA


# ======================
# DATA
# ======================
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}


for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mse = mean_squared_error(y_test, pred)
    results[name] = mse


# ======================
# GRADIENT COBRA
# ======================
cobra = GradientCOBRA(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    optimizer="grid",
    bandwidth_grid=np.linspace(0.1, 5.0, 15),
    optimizer_params={"verbose": True},
    random_state=42
)

cobra.fit(X_train, y_train)
cobra_pred = cobra.predict(X_test)

cobra_mse = mean_squared_error(y_test, cobra_pred)
results["GradientCOBRA"] = cobra_mse


# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON (lower is better) =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")


===== MSE COMPARISON (lower is better) =====

RandomForest         : 0.2537
GradientCOBRA        : 0.3009
Ridge                : 0.5558
LinearRegression     : 0.5559
SVR                  : 1.2197


# MixCobra

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

from cobra.mixcobra import MixCOBRARegressor

# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}

# train single models
for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = mean_squared_error(y_test, pred)

# ======================
# MIXCOBRA
# ======================
mix = MixCOBRARegressor(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    alpha_grid=np.linspace(0.1, 3.0, 10),
    beta_grid=np.linspace(0.1, 3.0, 10),
    random_state=42
)

mix.fit(X_train, y_train)
mix_pred = mix.predict(X_test)

results["MixCOBRA"] = mean_squared_error(y_test, mix_pred)

# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")


===== MSE COMPARISON =====

RandomForest         : 0.2537
MixCOBRA             : 0.3998
Ridge                : 0.5558
LinearRegression     : 0.5559
SVR                  : 1.2197


In [ ]:
X -> Splitter -> Estimators -> Projector